In [28]:
# 1. gdrive 연결
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [29]:
import os
import pandas as pd

# Google Drive의 기본 경로를 설정합니다.
# 필요에 따라 'My Drive' 부분을 사용자 Drive 경로에 맞게 수정하세요.
base_drive_path = '/content/gdrive/My Drive/data_filtering'

# 원본 파일이 있는 폴더와 파일을 저장할 폴더 경로를 설정합니다.
original_folder_path = os.path.join(base_drive_path, 'original')
output_folder_path = os.path.join(base_drive_path, 'filtered') # 사용자 요청에 따라 filtered 폴더에 저장

# 출력 폴더가 없으면 생성합니다.
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

print(f"원본 폴더: {original_folder_path}")
print(f"저장 폴더: {output_folder_path}")

원본 폴더: /content/gdrive/My Drive/data_filtering/original
저장 폴더: /content/gdrive/My Drive/data_filtering/filtered


In [30]:
# 2. original 폴더 안의 csv 파일 목록 가져오기
csv_files = [f for f in os.listdir(original_folder_path) if f.endswith('.csv')]

if not csv_files:
    print(f"'{original_folder_path}' 폴더에 CSV 파일이 없습니다.")
else:
    print(f"처리할 CSV 파일 목록: {csv_files}")

처리할 CSV 파일 목록: ['TEST_06.csv', 'TEST_03.csv', 'TEST_07.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_01.csv', 'TEST_08.csv', 'TEST_00.csv', 'TEST_02.csv', 'TEST_09.csv', 'train.csv']


In [34]:
# 3. 각 csv 파일을 읽어와 원본 그대로 filtered 폴더에 저장
from datetime import datetime # datetime 모듈 import

# --- 파일 처리 및 저장 루프 ---
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # 원본 파일명 그대로 저장

    print(f"\nProcessing {file_name}...")

    try:
        # CSV 파일 읽기 (utf-8 인코딩 지정)
        df = pd.read_csv(original_file_path, encoding='utf-8')
        print(f"Original shape: {df.shape}")

        # 1. 읽어오는게 무리가 없는지, 각 파일을 읽어올때마다 맨 위 한줄씩 출력(디버깅용)
        if not df.empty:
            print("First row (for debugging):")
            display(df.head(1)) # display를 사용하여 보기 좋게 출력
        else:
            print("DataFrame is empty.")

        # --- 컬럼명 변경 ---
        new_column_names = {
            '영업일자': 'date',
            '영업장명_메뉴명': 'store_menu',
            '매출수량': 'sales'
        }
        df = df.rename(columns=new_column_names)
        print("Columns renamed.")
        # --- 컬럼명 변경 끝 ---

        # --- store_menu 컬럼 분리 ---
        if 'store_menu' in df.columns:
            df[['store', 'menu']] = df['store_menu'].str.split('_', n=1, expand=True)
            print("'store_menu' column split into 'store' and 'menu'.")
        else:
            print("'store_menu' column not found.")
        # --- store_menu 컬럼 분리 끝 ---

        # --- date 열을 datetime 객체로 변환하고 dateordinal 열 추가 ---
        if 'date' in df.columns:
            # 'date' 컬럼을 datetime 형식으로 변환 (에러 발생 시 NaT 값으로 처리)
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            # datetime 객체를 ordinal 값으로 변환
            df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)
            print("'date' column converted to datetime and 'date_ordinal' column added.")
        else:
            print("'date' column not found.")
        # --- date 열 처리 끝 ---


        # --- 컬럼 순서 조정 (date_ordinal을 date 왼쪽으로, sales를 마지막으로) ---
        if 'date' in df.columns and 'date_ordinal' in df.columns and 'sales' in df.columns:
            cols = df.columns.tolist()
            # Remove date_ordinal and sales
            cols.remove('date_ordinal')
            cols.remove('sales')
            # Find the index of 'date'
            date_index = cols.index('date')
            # Insert date_ordinal before date
            cols.insert(date_index, 'date_ordinal')
            # Add sales to the end
            cols.append('sales')
            df = df[cols]
            print("'date_ordinal' moved left of 'date', 'sales' moved to the end.")
        elif 'sales' in df.columns: # If date_ordinal or date is not present, just move sales to the end
             cols = [col for col in df.columns if col != 'sales'] + ['sales']
             df = df[cols]
             print("'sales' column moved to the end.")
        elif 'date' in df.columns and 'date_ordinal' in df.columns: # If sales is not present, just move date_ordinal left of date
             cols = df.columns.tolist()
             cols.remove('date_ordinal')
             date_index = cols.index('date')
             cols.insert(date_index, 'date_ordinal')
             df = df[cols]
             print("'date_ordinal' moved left of 'date'.")
        # --- 컬럼 순서 조정 끝 ---


        # 원본 DataFrame을 filtered 폴더에 저장 (파일명 유지)
        print(f"Saving original {file_name} to {output_folder_path}...")
        df.to_csv(output_file_path, index=False)

        print(f"Saved original {file_name}")

    except FileNotFoundError:
        print(f"Error: File not found at {original_file_path}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

print("\nAll files processing attempt completed.")


Processing TEST_06.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-01-12,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
'date_ordinal' moved left of 'date', 'sales' moved to the end.
Saving original TEST_06.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved original TEST_06.csv

Processing TEST_03.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2024-09-29,느티나무 셀프BBQ_1인 수저세트,5


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
'date_ordinal' moved left of 'date', 'sales' moved to the end.
Saving original TEST_03.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved original TEST_03.csv

Processing TEST_07.csv...
Original shape: (5404, 3)
First row (for debugging):


...

Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
'date_ordinal' moved left of 'date', 'sales' moved to the end.
Saving original TEST_02.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved original TEST_02.csv

Processing TEST_09.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-04-27,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
'date_ordinal' moved left of 'date', 'sales' moved to the end.
Saving original TEST_09.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved original TEST_09.csv

Processing train.csv...
Original shape: (102676, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
'date_ordinal' moved left of 'date', 'sales' moved to the end.
Saving original train.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved original train.csv

All files processing attempt completed.
